# EDA na Prática

Nas aulas 9 e 10 vimos os quatro princípios da EDA (estrutural, distribucional, relacional e detecção de outliers/anomalias), como tratar dados ausentes e outliers, e como fazer análise descritiva, bivariada e multivariada. Hoje é a hora de **aplicarmos na prática**. Durante a primeira hora da aula, serão apresentadas as técnicas no dataset *Heart Disease* (UCI) e depois **cada aluno** irá aplicar as técnicas no dataset que foi escolhido semana passada.

*Relembrando*: a EDA não é uma linha reta

A EDA costuma ser apresentada como uma sequência de etapas (estrutura → tratamento → descrição → relação), mas na prática ela é **iterativa**: às vezes tratamos um outlier, olhamos a estatística descritiva de novo, e percebemos que precisa voltar atrás. Hoje vamos deixar isso explícito: olhamos a estatística descritiva **antes** de tratar dados ausentes/outliers (pra saber *como* tratar) e **depois** (pra confirmar o efeito do tratamento).

## Como funcionará a atividade

- **Duração:** ~1h40min
- **Formato:** individual
- **Passo 1:** acompanhem a demonstração no dataset Heart Disease.
- **Passo 2:** importem o dataset que vocês escolheram e enviaram anteriormente.
- **Passo 3:** repliquem o mesmo ciclo no dataset escolhido, respondendo às perguntas guiadas.
- **Uso de IA:** permitido, conforme o Plano de Ensino — documentem na seção final.

## Setup de Bibliotecas e Dados

In [1]:
print("oi")

oi


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

COR_NEUTRA = "lightgray"     # convenção: cinza-claro pra tudo que é "pano de fundo" genérico
COR_CONTEXTO = "#e9d4d0"    # rosa-claro pra pano de fundo / classe de referência
COR_DESTAQUE = "#2d223a"    # roxo pra classe de interesse / "o personagem da história"
COR_ALERTA = "tab:red"       # exceção deliberada: outliers/pontos de atenção

## Carregando o Dataset

In [3]:
import seaborn as sns
import pandas as pd

# Carrega o dataset do Titanic diretamente
df = sns.load_dataset('titanic')

# Ajusta a variável alvo para texto (facilita paletas de cores)
df['sobreviveu'] = df['survived'].map({0: 'Não', 1: 'Sim'})

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,sobreviveu
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,Não
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,Sim
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,Sim
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,Sim
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,Não


## 1. Princípio Estrutural

Antes de qualquer análise, precisamos entender a "anatomia" dos dados: quantas linhas e colunas temos, que tipo de dado cada coluna guarda, se há valores faltantes e se há duplicadas.

**Antes de rodar:** o Heart Disease tem 13 variáveis preditoras (idade, sexo, tipo de dor no peito, pressão em repouso, colesterol, entre outras) mais o alvo (`num`, presença/grau de doença cardíaca). Quantas linhas vocês esperam que um dataset clínico como esse tenha — dezenas, centenas ou milhares de pacientes?

### Dimensão e Tipos de Dados

In [4]:
df.shape

(891, 16)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
 15  sobreviveu   891 non-null    object  
dtypes: bool(2), category(2), float64(2), int64(4), object(6)
memory usage: 87.6+ KB


In [6]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


### Duplicadas

In [7]:
df.duplicated().sum()
df = df.drop_duplicates()

Removemos as duplicatas que deram mais que 100 de um total de 891 linhas por ser um número considerado alto

### Renomeando o atributo alvo para facilitar a interpretação

O dataset original registra survived como valores numéricos: 0 (não sobreviveu) e 1 (sobreviveu). Pra hoje, vamos transformar essa coluna em uma variável categórica mais legível, sobreviveu, já que é essa a distinção que vamos usar nas cores ao longo da atividade (rosa = não sobreviveu, roxo = sobreviveu).

In [8]:
import numpy as np

# Cria a nova variável categórica com nomes legíveis
df["sobreviveu"] = np.where(df["survived"] > 0, "Sobreviveu", "Não Sobreviveu")

# Exibe a contagem de cada classe
print(df["sobreviveu"].value_counts())

# Remove a coluna numérica antiga
df.drop(columns=["survived"], inplace=True)

# Exibe as primeiras linhas do dataset
df.head()

sobreviveu
Não Sobreviveu    461
Sobreviveu        323
Name: count, dtype: int64


,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,sobreviveu
0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,Não Sobreviveu
1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,Sobreviveu
2,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,Sobreviveu
3,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,Sobreviveu
4,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,Não Sobreviveu


## 2. Dados Faltantes

In [9]:
df.isna().sum()

pclass           0
sex              0
age            106
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           582
embark_town      2
alive            0
alone            0
sobreviveu       0
dtype: int64

**Discussao:** Alguma feature tem dados ausentes nos seus dados?






*Resposta*: Sim. As variáveis deck (582 valores ausentes), age (106 valores ausentes) e embarked / embark_town (2 valores ausentes) possuem dados faltantes na base.

#### Identificando o mecanismo de dados ausentes

No dataset atual temos as variáveis deck, age e embarked (embark_town) com dados ausentes.

Pela fonte de dados observamos que:

deck - Letra referente ao deck (pavimento/cabine) onde o passageiro estava hospedado.

age - Idade do passageiro em anos.

embarked / embark_town - Porto onde o passageiro realizou o embarque (Cherbourg, Queenstown ou Southampton).

Como os dados foram coletados:

O conjunto de dados foi compilado a partir de registros históricos de compra de bilhetes da companhia White Star Line, manifestos de embarque e documentos oficiais compilados por pesquisadores após o naufrágio de 1912.

Qual a natureza da fonte de dados:

deck (582 omissões): A designação exata de cabine/deck era registrada detalhadamente quase que de forma exclusiva para passageiros da 1ª classe. Os bilhetes da 2ª e 3ª classe raramente continham a numeração do camarote nos cartões de embarque, gerando essa ausência massiva de informação.

age (106 omissões): Em 1912, a coleta de dados de idade não era rigorosa na emissão de passagens marítimas, especialmente para famílias de imigrantes da 3ª classe vindo de múltiplos países da Europa.

embarked (2 omissões): Apenas dois passageiros não possuem a confirmação do porto registrada, o que representa uma perda acidental irrelevante nos arquivos históricos.

**As ausencias ocorrem juntas?**

Contar ausentes por coluna não conta a história toda: se duas colunas ficam ausentes **nas mesmas linhas**, isso é um indício de padrão sistemático (ex: um exame que não foi feito, afetando várias medidas de uma vez).



In [10]:
cols_com_ausentes = df.columns[df.isna().any()].tolist()
print("Colunas com ausentes:", cols_com_ausentes)
print(df[cols_com_ausentes].isna().sum())


Colunas com ausentes: ['age', 'embarked', 'deck', 'embark_town']
age            106
embarked         2
deck           582
embark_town      2
dtype: int64


Com poucos ausentes (no máximo poucas dezenas de linhas), uma correlação entre as colunas de ausência pode parecer precisa, mas na prática é calculada em cima de um poucos de casos, ou seja, número pequeno demais pra confiar no coeficiente. Aqui a contagem bruta (quantas linhas têm as duas ausências ao mesmo tempo) é mais honesta que uma correlação.

In [11]:
tabela_contingencia = pd.crosstab(df[cols_com_ausentes[0]].isna(), df[cols_com_ausentes[2]].isna(),
            rownames=[f"{cols_com_ausentes[0]} ausente"], colnames=[f"{cols_com_ausentes[2]} ausente"])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(tabela_contingencia, annot=True, fmt="d", cmap="magma", ax=ax)
ax.set_title("As ausências ocorrem juntas?")
plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\4269708942.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Mecanismo de Ausência Predominante: MAR (Missing at Random / Viés Correlacionado).

Justificativa: As ausências não são totalmente aleatórias (descarte o MCAR). Dos 106 passageiros com idade faltante, 82% (87 passageiros) também não têm o registro do deck.

Explicação pelo Conhecimento de Domínio: Existe uma dependência direta com a classe da passagem (pclass). Passageiros da 3ª classe (imigrantes e passagens populares) raramente tinham número de cabine no bilhete (deck nulo) e frequentemente não tinham documentação formal de idade registrada no manifesto de embarque (age nulo). Como a ausência pode ser explicada por outras variáveis observadas na base (como a classe ou valor da tarifa), trata-se de um mecanismo MAR.

## 3. Princípio Distribucional

Ainda com os dados originais (sem tratar ausentes ou outliers), vamos calcular as estatísticas descritivas e olhar as distribuições. Esse retrato "antes" é o que vai *orientar as decisões de tratamento* na próxima seção. Por exemplo, se uma variável for muito assimétrica, faz mais sentido imputar com a mediana do que com a média.

In [12]:

VARS_QUANT = ["age", "fare", "sibsp", "parch", "pclass"]
VARS_QUALI = ["sex", "embarked", "class", "who", "adult_male", "deck", "embark_town", "alive", "alone", "sobreviveu"]


### Tabela descritiva, com assimetria e curtose
desc = df[VARS_QUANT].describe().T
desc["skewness"] = df[VARS_QUANT].skew()
desc["kurtosis"] = df[VARS_QUANT].kurtosis()

(
    desc[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]]
    .rename(columns={"50%": "median"})
    .round(2)
    .style.format(precision=2)
    .background_gradient(cmap="magma", subset=["skewness", "kurtosis"])
)

,mean,median,std,min,max,skewness,kurtosis
age,29.87,28.25,14.76,0.42,80.00,0.37,0.09
fare,34.71,15.90,52.16,0.00,512.33,4.57,30.21
sibsp,0.52,0.00,0.99,0.00,8.00,3.06,12.76
parch,0.42,0.00,0.84,0.00,6.00,2.63,8.97
pclass,2.24,3.00,0.86,1.00,3.00,-0.49,-1.46


Variáveis Identificadas

fare: Skewness = 4.57 | Kurtosis = 30.21 (assimetria extrema e cauda muito pesada).

sibsp: Skewness = 3.06 | Kurtosis = 12.76 (alta assimetria).

parch: Skewness = 2.63 | Kurtosis = 8.97 (alta assimetria).

Coerência com o Conhecimento de Domínio


fare (Tarifa): A grande maioria dos passageiros viajava na 3ª e 2ª classe com bilhetes baratos, enquanto uma pequena minoria de milionários da 1ª classe pagou tarifas exorbitantes (até 512 libras). Esse grupo restrito puxa a cauda fortemente para a direita, fazendo a média (34.71) ser mais que o dobro da mediana (15.90).

sibsp e parch (Família a bordo): A maioria viajava sozinha ou acompanhada de no máximo uma pessoa (mediana = 0). Famílias muito numerosas (5 a 8 parentes) eram exceções pontuais que geraram a cauda longa.

age (Idade): Comportou-se de forma oposta aos demais, apresentando uma distribuição próxima da normal (skew = 0.37 e kurt = 0.09), refletindo uma amostragem natural de viajantes (crianças, jovens e idosos).

### Distribuições de dados

Onde fizer sentido, comparamos "Sem doença" (rosa) vs. "Com doença" (roxo). Vamos utilizar esta convenção de cor que vamos manter até a análise bivariada.


In [13]:
# Mapeamento das variáveis quantitativas do Titanic
VARS_QUANT = ["age", "fare", "sibsp", "parch"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=False)
axes = axes.ravel()

for i, col in enumerate(VARS_QUANT):
    sns.histplot(data=df, x=col, kde=True, color=COR_NEUTRA, ax=axes[i])
    axes[i].set_title(col)

for ax in axes[len(VARS_QUANT):]:
    ax.axis("off")

fig.suptitle("Distribuição das Variáveis Quantitativas: Titanic (sem separar por classe)", fontsize=14)
plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\2095175372.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
fig, axes = plt.subplots(len(VARS_QUANT), 2, figsize=(10, 3.5 * len(VARS_QUANT)), sharey="row")

for i, col in enumerate(VARS_QUANT):
    lim = (df[col].min(), df[col].max())
    for j, (classe, cor) in enumerate([("Não Sobreviveu", COR_CONTEXTO), ("Sobreviveu", COR_DESTAQUE)]):
        subset = df[df["sobreviveu"] == classe]
        sns.histplot(data=subset, x=col, kde=True, color=cor, ax=axes[i, j])
        axes[i, j].set_xlim(lim)
        axes[i, j].set_title(f"{col} — {classe}")

fig.suptitle("Distribuição das Variáveis Quantitativas: Titanic (por classe, em painéis separados)", fontsize=14, y=1.001)
plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\2681875986.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Para discutir: Comparação de distribuições entre as classes Ao observar os painéis lado a lado com a mesma escala de eixos, as variáveis fare e age revelam as mudanças de formato mais marcantes entre os dois grupos: fare (Tarifa): A diferença é a mais evidente. No painel de "Não Sobreviveu", a massa de dados está quase totalmente espremida nos valores mais baixos (próximos de zero, típicos da 3ª classe). Já no painel de "Sobreviveu", a distribuição possui uma cauda visivelmente mais longa e densa para a direita, mostrando que a probabilidade de sobrevivência cresce conforme a tarifa aumenta.age (Idade): O grupo "Não Sobreviveu" tem um p**Pra discutir: Comparação de distribuições entre as classes**

Ao observar os painéis lado a lado com a mesma escala de eixos, as variáveis **`fare`** e **`age`** revelam as mudanças de formato mais marcantes entre os grupos:

* **`fare` (Tarifa):** A diferença mais evidente do dataset. No grupo **Não Sobreviveu**, a massa de dados está quase totalmente concentrada nos valores mais baixos (próximos de zero, típicos da 3ª classe). No grupo **Sobreviveu**, a distribuição possui uma cauda visivelmente mais longa e densa à direita, evidenciando que a probabilidade de sobrevivência cresce conforme o valor da tarifa aumenta.
* **`age` (Idade):** O grupo **Não Sobreviveu** exibe um pico acentuado focado em jovens adultos (entre 20 e 35 anos). Já no grupo **Sobreviveu**, a curva altera seu formato e ganha um novo pico na extrema esquerda (crianças de 0 a 10 anos), retratando graficamente a aplicação do protocolo histórico *"mulheres e crianças primeiro"*.
* **`sibsp` e `parch` (Relações Familiares):** O grupo que não sobreviveu concentra-se nos extremos: pessoas viajando completamente sozinhas (valor 0) ou em famílias muito numerosas (5 ou mais parentes). Em contrapartida, entre os sobreviventes nota-se uma proporção significativamente maior de passageiros acompanhados por pequenos núcleos familiares (1 a 2 parentes).ico muito bem definido focado em jovens adultos (entre 20 e 35 anos). No grupo "Sobreviveu", a curva altera o seu formato e ganha um pico na extrema esquerda (crianças de 0 a 10 anos), refletindo graficamente o protocolo histórico de evacuação "mulheres e crianças primeiro".sibsp e parch (Relações Familiares): Quem não sobreviveu concentra-se fortemente no valor $0$ (pessoas viajando sozinhas) ou nos extremos altos ($5+$ parentes). Já nos sobreviventes, há uma proporção maior de passageiros viajando em pequenos núcleos de 1 a 2 parentes.

### Testes de Aderência
Será que a idade dos passageiros segue uma distribuição normal?

O Shapiro-Wilk é o teste de aderência mais usado para normalidade, junto com o QQ-Plot como visualização de apoio. Vamos testar com age.

In [15]:
stat_sw, p_valor_sw = stats.shapiro(df["age"].dropna())
print(f"Shapiro-Wilk — age: estatística={stat_sw:.4f}, p-valor={p_valor_sw:.4g}")
print("Rejeita H0 (normalidade)?", "Sim" if p_valor_sw < 0.05 else "Não", "(alpha = 0.05)")

Shapiro-Wilk — age: estatística=0.9829, p-valor=4.172e-07
Rejeita H0 (normalidade)? Sim (alpha = 0.05)


**Como ler:** se os pontos (roxo) seguissem a reta vermelha de perto, a variável seguiria uma normal. Pontos que se afastam da reta nas pontas indicam cauda mais pesada ou mais leve que a normal — coerente com o resultado do Shapiro-Wilk acima.

**Nota: cuidado com duas armadilhas de interpretação**

- **Tamanho da amostra e poder do teste:** é comum achar que "poucos dados" tornam mais fácil rejeitar a normalidade e é o contrário. Amostras **pequenas** têm **menos poder estatístico**, ou seja, tendem a *não* rejeitar mesmo quando os dados realmente não são normais. Já amostras **muito grandes** (milhares de linhas) deixam o teste sensível demais, rejeitando por desvios triviais. Com n≈303 (nem pequeno, nem enorme) e três evidências concordando — skewness alto, QQ-Plot afastado da reta, e Shapiro-Wilk rejeitando — o resultado é razoavelmente confiável, não um artefato do tamanho da amostra.
- **Como afirmar o resultado:** quando o teste **rejeita** H0 (nosso caso, p < 0.05), a afirmação correta é "temos evidência estatística **contra** a normalidade" que é uma conclusão forte. A frase "não podemos garantir normalidade" cabe no caso oposto: quando o teste **não rejeita** H0, já que a ausência de rejeição nunca prova que os dados são normais, só que não há evidência suficiente contra.

In [16]:
resultados_shapiro = []
for col in VARS_QUANT:
    stat, p = stats.shapiro(df[col].dropna())
    resultados_shapiro.append({"variavel": col, "estatistica": round(stat, 4),
                                "p_valor": round(p, 6), "rejeita_normalidade": p < 0.05})

pd.DataFrame(resultados_shapiro).set_index("variavel")

,estatistica,p_valor,rejeita_normalidade
variavel,,,
age,0.9829,0.0,True
fare,0.5375,0.0,True
sibsp,0.5691,0.0,True
parch,0.5554,0.0,True


ECDF — outra forma de ver a distribuição

Diferente do histograma, a ECDF não depende de escolher o número de bins: para cada valor no eixo X, a curva mostra a proporção acumulada de observações menores ou iguais àquele valor. É especialmente útil pra comparar duas distribuições lado a lado — onde as curvas se distanciam mais é onde a diferença entre os grupos é mais perceptível.

Vamos ilustrar com uma única variável (fare, tarifa paga pelo bilhete), só pra fixar como essa visualização se lê.

Python

In [17]:
fig, ax = plt.subplots(figsize=(8, 5))

# Plotagem da ECDF para a variável 'fare'
sns.ecdfplot(data=df, x="fare", color=COR_NEUTRA, ax=ax)

ax.set_title("ECDF: Distribuição Acumulada da Tarifa (fare)")
ax.set_xlabel("Tarifa (fare)")
ax.set_ylabel("Proporção Acumulada")

plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\3696481212.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Como ler:** em qualquer ponto do eixo X, a altura da curva diz "que fração dos passageiros desse grupo tem fare até aqui". Onde as duas curvas (rosa e roxa) se distanciam mais, ali está a faixa de fare onde os dois grupos mais diferem, sem precisar decidir uma largura de bin como no histograma.

### Métricas de Tendêncial Central

In [18]:
media_fare = df["fare"].mean()
mediana_fare = df["fare"].median()

plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="fare", kde=True, color=COR_NEUTRA)
plt.axvline(media_fare, color=COR_ALERTA, linestyle="--", linewidth=2, label=f"Média = {media_fare:.1f}")
plt.axvline(mediana_fare, color="black", linestyle="-", linewidth=2, label=f"Mediana = {mediana_fare:.1f}")
plt.title("Tarifa (fare): Distribuição com Média e Mediana")
plt.legend()
plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\2899104218.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Dispersão

In [19]:
# Tabela de medidas de dispersão
dispersao = df[VARS_QUANT].agg(["var", "std"]).T
dispersao["IQR"] = df[VARS_QUANT].quantile(0.75) - df[VARS_QUANT].quantile(0.25)
dispersao.round(2)

### Boxplot e violin plot por classe de sobrevivência
palette_classe = {"Não Sobreviveu": COR_CONTEXTO, "Sobreviveu": COR_DESTAQUE}

fig, axes = plt.subplots(2, len(VARS_QUANT), figsize=(4 * len(VARS_QUANT), 9), sharex="col")

for i, col in enumerate(VARS_QUANT):
    sns.boxplot(
        data=df,
        x="sobreviveu",
        y=col,
        hue="sobreviveu",
        palette=palette_classe,
        legend=False,
        ax=axes[0, i]
    )
    axes[0, i].set_title(col)
    axes[0, i].set_xlabel("")

    sns.violinplot(
        data=df,
        x="sobreviveu",
        y=col,
        hue="sobreviveu",
        palette=palette_classe,
        legend=False,
        ax=axes[1, i]
    )
    axes[1, i].set_xlabel("")

axes[0, 0].set_ylabel("Boxplot")
axes[1, 0].set_ylabel("Violin plot")
fig.suptitle("Dispersão por Classe (Sobrevivência) — Boxplot vs. Violin Plot", fontsize=14)
plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\1788464128.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Para discutir: Diferenças de IQR e distribuição entre as classes**

Sim, a variável **`fare`** (e em menor escala, **`parch`**) apresenta uma mudança drástica de posição e amplitude da caixa (IQR) entre os dois grupos, oferecendo um forte indício prévio de seu poder preditivo:

* **`fare` (Tarifa) — Diferença gritante:**
* **Não Sobreviveu:** A caixa (IQR) está quase inteiramente comprimida na base (entre ~7 e 26), com a mediana muito próxima de zero.
* **Sobreviveu:** A caixa desloca-se visivelmente para cima (IQR de ~12 a 55), com mediana mais que o dobro em relação aos não-sobreviventes. O *Violin Plot* confirma essa barriga bem mais encorpada em valores mais altos de passagem.


* **`parch` (Pais e Filhos a bordo) — Expansão do IQR:**
* **Não Sobreviveu:** A caixa nem sequer abre; Q1, mediana e Q3 estão todos colados no $0$ (a imensa maioria viajava sem pais/filhos e não sobreviveu).
* **Sobreviveu:** O IQR se expande até o valor $1$, mostrando que viajar acompanhado de pequenos núcleos familiares esteve associado a maior taxa de sobrevivência.


* **`age` (Idade) — O detalhe no Violin Plot:**
* Embora as caixas (IQR) tenham alturas e medianas parecidas (mediana por volta dos 28 anos em ambos), o **Violin Plot** captura o que o Boxplot esconde: a classe **"Sobreviveu"** possui uma protuberância marcante na base (0 a 10 anos), evidenciando a prioridade dada às crianças nos botes.

## 4. Tratamento de Dados Ausentes e Outliers

Agora que já vimos o retrato estatístico "antes" (Seção 3), temos a informação que faltava pra decidir como tratar tanto os dados ausentes (`ca`, `thal`) quanto os outliers das variáveis quantitativas. Depois do tratamento, vamos rodar a estatística descritiva mais uma vez ("depois") pra confirmar o efeito das escolhas e fechar o ciclo iterativo que começamos lá na abertura.

### Detecção de Outliers por IQR

A regra clássica marca como outlier qualquer ponto fora de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.

In [20]:
def limites_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

resumo_outliers = {}
for col in VARS_QUANT:
    lim_inf, lim_sup = limites_iqr(df[col])
    n_outliers = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    resumo_outliers[col] = {
        "limite_inferior": round(lim_inf, 2),
        "limite_superior": round(lim_sup, 2),
        "n_outliers": n_outliers,
    }

pd.DataFrame(resumo_outliers).T

,limite_inferior,limite_superior,n_outliers
age,-8.50,67.5,7.0
fare,-31.04,73.2,102.0
sibsp,-1.50,2.5,39.0
parch,-1.50,2.5,15.0


### Visualizando os Outliers

In [21]:
fig, axes = plt.subplots(1, len(VARS_QUANT), figsize=(4 * len(VARS_QUANT), 5))

for i, col in enumerate(VARS_QUANT):
    sns.boxplot(data=df, y=col, color=COR_NEUTRA,
                flierprops=dict(markerfacecolor=COR_ALERTA, markeredgecolor=COR_ALERTA, marker="o"),
                ax=axes[i])
    axes[i].set_title(col)

fig.suptitle("Outliers pela regra do IQR (pontos em vermelho)", fontsize=14)
plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\2858582539.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Diagnóstico dos Outliers**

Todos os pontos identificados enquadram-se como **Outliers Estatísticos (Legítimos)**. Nenhum valor viola restrições de domínio ou decorre de erro de digitação, refletindo fielmente a realidade demográfica e socioeconômica de 1912.

* **`fare` (102 outliers | Limite Superior: 73.20 | Máximo: £512.33):** Totalmente plausíveis do ponto de vista histórico. Correspondem a suítes de altíssimo luxo na 1ª classe e a bilhetes comprados em grupo. Como a riqueza foi um fator decisivo no acesso aos botes, devem ser **mantidos** e tratados via transformação logarítmica ($\log(1 + \text{fare})$) ou binning (faixas de preço).
* **`age` (7 outliers | Limite Superior: 67.5 | Máximo: 80 anos):** Plausíveis. O passageiro registrado com 80 anos (Algernon Barkworth) de fato esteve no navio e sobreviveu. **Manter** na base sem alterações.
* **`sibsp` (39 outliers) e `parch` (15 outliers):** Representam famílias numerosas da época viajando juntas (ex: a família Sage, de 11 pessoas na 3ª classe). **Manter** ou simplificar criando uma variável agregada (`tamanho_familia = sibsp + parch + 1`).

---

**Decisão de Tratamento**

**Manter e Transformar**. A eliminação dessas observações descartaria exatamente os passageiros mais abastados e as maiores famílias — dois dos perfis de maior relevância para explicar a taxa de sobrevivência.

Decidindo o Tratamento dos Dados Ausentes

Reunindo o que analisamos sobre as variáveis da base do Titanic com o seu total atual de registros:

embarked (MCAR, ~0,26% ausente): Como temos apenas 2 linhas faltantes em um total de 784 registros, a remoção direta dessas observações (ajustando a base para 782 linhas) é a escolha mais pragmática — o risco de introduzir viés é nulo e o impacto estatístico na amostra é insignificante.

age (MAR, ~20% ausente): A ausência está correlacionada à classe social (pclass). Deletar essas observações descartaria uma parcela considerável da base; por isso, a decisão é imputar a mediana da idade calculada pelo grupo de classe/sexo.

deck / cabin (MAR/MNAR, ~77% ausente): Pelo volume massivo de dados faltantes, qualquer remoção de linhas destruiria a amostragem. A estratégia ideal é descartar a coluna original do modelo ou tratar a ausência como uma nova categoria explícita ("Desconhecido").

In [22]:
linhas_antes = df.shape[0]
df_antes = df.copy()

# Remoção das poucas linhas com dados ausentes em 'embarked' (apenas 2 registros)
df = df.dropna(subset=["embarked"]).reset_index(drop=True)
linhas_depois = df.shape[0]

print(f"Linhas antes: {linhas_antes} | Linhas depois: {linhas_depois} | Removidas: {linhas_antes - linhas_depois}")

Linhas antes: 784 | Linhas depois: 782 | Removidas: 2


### Decidindo o Tratamento dos Outliers
Diferente dos dados ausentes, aqui não vamos remover observações. Pelos valores encontrados (dentro de faixas historicamente plausíveis, como bilhetes de luxo da 1ª classe e passageiros idosos), a classificação é "Estatístico" — outliers legítimos, não erros de digitação. Sendo assim, o tratamento recomendado é manter (a informação é real e socioeconomicamente relevante — por exemplo, tarifas altas refletem a 1ª classe, determinante na taxa de sobrevivência) ou transformar, caso necessário para técnicas sensíveis à assimetria.

Vamos manter os valores originais por enquanto, mas ilustrar como ficaria uma transformação logarítmica em fare — útil caso, em etapas futuras de modelagem, a assimetria acentuada atrapalhe algum algoritmo.

In [23]:
df["fare_log"] = np.log1p(df["fare"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(data=df, x="fare", kde=True, color=COR_NEUTRA, ax=axes[0])
axes[0].set_title(f"fare original (skew={df['fare'].skew():.2f})")

sns.histplot(data=df, x="fare_log", kde=True, color=COR_NEUTRA, ax=axes[1])
axes[1].set_title(f"log(1 + fare) (skew={df['fare_log'].skew():.2f})")

plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\349991150.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


a transformação logarítmica reduziu drasticamente a assimetria visual e estatística da variável.

Redução numérica (skewness): O valor de skewness caiu de 4,58 (forte assimetria à direita) para apenas 0,42 (próximo de uma distribuição simétrica).

Mudança visual: A distribuição original estava totalmente concentrada nos valores baixos perto de zero, com uma cauda longa que se estendia até 500. Após aplicar o log(1+fare), os valores foram comprimidos e redistribuídos, resultando em um formato muito mais centralizado e próximo de uma curva normal.

## 5. Estatística Descritiva pt 2

Fechando o ciclo iterativo: tratamos os ausentes e decidimos manter os outliers. Agora rodamos a mesma bateria de estatísticas da Seção 3 para comparar diretamente com o "antes".

In [24]:
desc_depois = df[VARS_QUANT].describe().T
desc_depois["skewness"] = df[VARS_QUANT].skew()
desc_depois["kurtosis"] = df[VARS_QUANT].kurtosis()

(
    desc_depois[["mean", "50%", "std", "min", "max", "skewness", "kurtosis"]]
    .rename(columns={"50%": "median"})
    .round(2)
    .style.format(precision=2)
    .background_gradient(cmap="Blues", subset=["skewness", "kurtosis"])
)

,mean,median,std,min,max,skewness,kurtosis
age,29.81,28.00,14.73,0.42,80.00,0.37,0.10
fare,34.60,15.88,52.18,0.00,512.33,4.58,30.29
sibsp,0.52,0.00,0.99,0.00,8.00,3.05,12.73
parch,0.42,0.00,0.84,0.00,6.00,2.63,8.94


### Comparando "antes"x "depois""



In [25]:
comparacao = pd.DataFrame({
    "skewness_antes": desc["skewness"],
    "skewness_depois": desc_depois["skewness"],
    "kurtosis_antes": desc["kurtosis"],
    "kurtosis_depois": desc_depois["kurtosis"],
})
comparacao["delta_skewness"] = (comparacao["skewness_depois"] - comparacao["skewness_antes"]).round(3)
comparacao["delta_kurtosis"] = (comparacao["kurtosis_depois"] - comparacao["kurtosis_antes"]).round(3)
comparacao.round(3)

,skewness_antes,skewness_depois,kurtosis_antes,kurtosis_depois,delta_skewness,delta_kurtosis
age,0.367,0.368,0.091,0.100,0.000,0.009
fare,4.571,4.583,30.208,30.292,0.012,0.083
parch,2.633,2.629,8.967,8.938,-0.004,-0.029
pclass,-0.489,NaN,-1.459,NaN,NaN,NaN
sibsp,3.055,3.051,12.762,12.728,-0.004,-0.034


Os valores praticamente não mudaram, e esse resultado era totalmente esperado:

Variação irrisória: As diferenças nos índices de assimetria (skewness) e achatamento (kurtosis) foram insignificantes, alterando apenas a terceira casa decimal (deltas próximos de zero).

Consistência com a decisão: Como foram removidas apenas 2 linhas (~0,25% da base) e todos os outliers foram mantidos, a estrutura e a forma da distribuição dos dados permaneceram preservadas.

Confirmação de tratamento conservador: O resultado valida que a limpeza foi conservadora e segura. O procedimento cumpriu seu papel de sanar os dados ausentes sem distorcer as propriedades estatísticas originais da base.

Rodar o Fare nos dados tratados

In [26]:
# Teste de Shapiro-Wilk para 'fare' pré e pós-tratamento
stat_sw, p_valor_sw = stats.shapiro(df_antes["fare"].dropna())
stat_sw_depois, p_valor_sw_depois = stats.shapiro(df["fare"].dropna())

print(f"Shapiro-Wilk (antes)  — fare: estatística={stat_sw:.4f}, p-valor={p_valor_sw:.4g}")
print(f"Shapiro-Wilk (depois) — fare: estatística={stat_sw_depois:.4f}, p-valor={p_valor_sw_depois:.4g}")

Shapiro-Wilk (antes)  — fare: estatística=0.5375, p-valor=4.464e-41
Shapiro-Wilk (depois) — fare: estatística=0.5353, p-valor=4.188e-41


**Visualizando antes x depois, lado a lado**

Vamos sobrepor as distribuições de duas variáveis que mais chamaram atenção (`fare` e `age`), comparando a versão original (`df_antes`) com a tratada (`df`).

In [27]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col in zip(axes, ["fare", "age"]):
    sns.kdeplot(df_antes[col], color=COR_CONTEXTO, fill=True, label="Antes", ax=ax)
    sns.kdeplot(df[col], color=COR_DESTAQUE, fill=True, label="Depois", ax=ax)
    ax.set_title(col)
    ax.legend()

fig.suptitle("Distribuição Antes x Depois do Tratamento", fontsize=14)
plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\2729470280.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Sim, as curvas "antes" e "depois" se sobrepõem perfeitamente.

Por que isso é esperado: Removemos apenas 2 linhas (menos de 0,3% da base) e optamos por preservar os outliers legítimos. A estrutura e a forma original das distribuições de fare e age permaneceram intactas.

Validação do tratamento: Essa sobreposição confirma que o procedimento foi conservador e seguro, corrigindo dados ausentes pontuais sem distorcer o comportamento estatístico da amostra.

Importância do teste: A comparação visual "antes x depois" funciona como um sanity check fundamental na EDA. Ela garante, com evidências empíricas, que o tratamento não introduziu vieses ou alterações indesejadas na base de dados.

## 6. Princípio Relacional

### Análise de Variáveis Qualitativas

Agora focaremos na avaliação de frequência e desbalanceamento das variáveis qualitativas.




#### Frequências absolutas e relativas

In [28]:
for col in VARS_QUALI:
    freq_abs = df[col].value_counts()
    freq_rel = df[col].value_counts(normalize=True).round(3)
    print(f"\\n--- {col} ---")
    print(pd.concat([freq_abs.rename("absoluta"), freq_rel.rename("relativa")], axis=1))

\n--- sex ---
        absoluta  relativa
sex                       
male         491     0.628
female       291     0.372
\n--- embarked ---
          absoluta  relativa
embarked                    
S              568     0.726
C              155     0.198
Q               59     0.075
\n--- class ---
        absoluta  relativa
class                     
Third        405     0.518
First        212     0.271
Second       165     0.211
\n--- who ---
       absoluta  relativa
who                      
man         451     0.577
woman       249     0.318
child        82     0.105
\n--- adult_male ---
            absoluta  relativa
adult_male                    
True             451     0.577
False            331     0.423
\n--- deck ---
      absoluta  relativa
deck                    
C           59     0.295
B           44     0.220
D           33     0.165
E           32     0.160
A           15     0.075
F           13     0.065
G            4     0.020
\n--- embark_town ---
            

#### Visualizando as frequências

In [29]:
n_vars = len(VARS_QUALI)
fig, axes = plt.subplots(1, n_vars, figsize=(4 * n_vars, 4), sharey=True)


if n_vars == 1:
    axes = [axes]
else:
    axes = axes.ravel()

for i, col in enumerate(VARS_QUALI):
    if col in df.columns:
        df[col].value_counts().sort_index().plot(kind="bar", color=COR_NEUTRA, ax=axes[i])
        axes[i].set_title(col)
        axes[i].tick_params(axis="x", rotation=0)
    else:
        print(f"Atenção: coluna '{col}' não encontrada no DataFrame.")

fig.suptitle("Frequências das Variáveis Qualitativas", fontsize=14)
plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\3666637961.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Sim, a variável embarked (e sua duplicata embark_town) é a mais desbalanceada, acompanhada por deck e class:

embarked / embark_town: O porto de Southampton (S) concentra mais de 70% das observações (~570 passageiros), enquanto Queenstown (Q) representa menos de 10% da amostra.

deck: Apresenta barras extremamente baixas porque cerca de 77% da coluna é formada por dados ausentes.

class e sex: Apresentam desbalanço moderado, com a 3ª classe (Third) e o sexo masculino (male) dominando suas respectivas categorias.

Impacto na Análise Bivariada:
Cruzamentos com categorias muito reduzidas (como o porto Q ou a categoria child) geram subgrupos com pouquíssimas amostras. Isso causa volatilidade estatística — pequenos grupos podem apresentar taxas de sobrevivência aparentemente extremas (muito altas ou muito baixas) apenas por ruído amostral, e não por um padrão real

#### Desbalanceamento entre Classes

- **Razão de desbalanceamento** — # classe majoritária / # classe minoritária (quanto maior, mais desbalanceado).
- **Entropia de Shannon normalizada** — 1 = perfeitamente balanceado, 0 = uma classe domina tudo.

Vamos aplicar as duas em todas as qualitativas, incluindo o próprio alvo `sobreviveu`.

In [30]:
from scipy.stats import entropy

def metrica_desbalanceamento(serie):
    contagens = serie.value_counts()
    razao = contagens.max() / contagens.min() if contagens.min() > 0 else np.nan
    props = contagens / contagens.sum()
    n_classes = len(contagens)
    ent_normalizada = entropy(props, base=2) / np.log2(n_classes) if n_classes > 1 else np.nan
    return pd.Series({
        "n_classes": n_classes,
        "razao_desbalanceamento": round(razao, 2),
        "entropia_normalizada": round(ent_normalizada, 3),
    })

# Substituição da variável alvo para o Titanic ('sobreviveu' ou 'survived')
alvo = "sobreviveu" if "sobreviveu" in df.columns else "survived"
vars_para_checar = [col for col in VARS_QUALI if col != alvo] + [alvo]

pd.DataFrame({col: metrica_desbalanceamento(df[col]) for col in vars_para_checar if col in df.columns}).T

,n_classes,razao_desbalanceamento,entropia_normalizada
sex,2.0,1.69,0.952
embarked,3.0,9.63,0.681
class,3.0,2.45,0.931
who,3.0,5.50,0.836
adult_male,2.0,1.36,0.983
deck,7.0,14.75,0.891
embark_town,3.0,9.63,0.681
alive,2.0,1.44,0.977
alone,2.0,1.31,0.987
sobreviveu,2.0,1.44,0.977


### Análise do Desbalanceamento das Variáveis

Variáveis com menor entropia: Nenhuma variável está criticamente próxima de 0, mas embarked / embark_town apresenta a menor entropia normalizada (0,681) e uma alta razão de desbalanceamento (9,63), confirmando a forte dominância do porto de Southampton.

Avaliação da variável alvo (sobreviveu): O alvo está excelentemente balanceado:

Entropia: 0,977 (praticamente próxima do valor ideal 1).

Razão de desbalanceamento: Apenas 1,44 (~60% não sobreviveu vs. ~40% sobreviveu).

Conclusão: A distribuição do alvo é ideal. Ela garante estatísticas confiáveis na análise bivariada e dispensa a necessidade de técnicas de reamostragem (como SMOTE ou undersampling) na etapa de modelagem preditiva.

### Análise Bivariada

Agora começamos a avaliar como as variáveis de fato se relacionam entre si e com o alvo 'doença cardíaca'.

#### Quantitativa x Quantitativa - Correlação

Conhecemos três técnicas de correlação principais:
* **Pearson** (linear, sensível a outliers, exige normalidade),
* **Spearman** e **Kendall** (relação monotônica, não exigem normalidade).

Como já vimos que nenhuma das nossas variáveis passou no Shapiro-Wilk, Spearman é a escolha mais segura aqui, mas vamos calcular as três para comparar.

In [31]:
corr_pearson = df[VARS_QUANT].corr(method="pearson")
corr_spearman = df[VARS_QUANT].corr(method="spearman")
corr_kendall = df[VARS_QUANT].corr(method="kendall")

fig, ax = plt.subplots(1,3,figsize=(14, 4),sharey=True)
sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[0])
ax[0].set_title("Matriz de Correlação — Pearson")
sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[1])
ax[1].set_title("Matriz de Correlação — Spearman")
sns.heatmap(corr_kendall, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax[2])
ax[2].set_title("Matriz de Correlação — Kendall")
plt.tight_layout()
plt.show()


C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\3399575423.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Como os dados do triângulo superior e inferior são simétricos, poderíamos plotar a matriz de correlação da seguinte forma.

In [32]:
mask = np.triu(np.ones_like(corr_spearman, dtype=bool))

plt.figure(figsize=(4, 3))
sns.heatmap(corr_spearman, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", square=True, vmin=-1, vmax=1,)
plt.grid(False)

plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\1229768662.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Sim, Pearson e Spearman divergem bastante nos pares que envolvem a variável fare:

fare x sibsp: Pearson = 0,14 vs. Spearman = 0,41

fare x parch: Pearson = 0,19 vs. Spearman = 0,38

O que essa divergência indica:

Relação não-linear (monotônica): O aumento significativo no coeficiente de Spearman confirma que existe uma relação ordinal clara (famílias maiores tendem a pagar tarifas totais mais altas), mas essa associação não ocorre em linha reta.

Sensibilidade do Pearson à assimetria: O método de Pearson é fortemente afetado pela assimetria e pelos valores extremos de fare, subestimando a associação. Já o Spearman, por trabalhar com postos/rankings e não com os valores brutos, consegue capturar a real força do relacionamento entre essas variáveis.

#### Dispersão (scatter) entre duas variáveis quantitativas

Escolhemos sibsp x parch por um motivo de domínio e estrutura de dados: existe uma relação direta na composição familiar a bordo, onde passageiros viajando com cônjuges ou irmãos (sibsp) frequentemente também acompanham pais ou filhos (parch), tornando esse par ideal para visualizar essa dinâmica.

Nota para a análise do Titanic: essa escolha também é sustentada pela estatística, pois esse par apresentou a maior correlação registrada na matriz de Spearman (0,42).

In [33]:
plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=df,
    x="sibsp",
    y="parch",
    hue="sobreviveu",
    palette={"Não Sobreviveu": COR_CONTEXTO, "Sobreviveu": COR_DESTAQUE},
    alpha=0.8,
)
plt.title("Irmãos/Cônjuges (sibsp) x Pais/Filhos (parch)")
plt.xlabel("Irmãos / Cônjuges a bordo (sibsp)")
plt.ylabel("Pais / Filhos a bordo (parch)")
plt.tight_layout()
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\445659367.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


###  Tendência Visível
 Não há uma inclinação linear contínua, mas existe um padrão de limite bem definido: à medida que os valores de sibsp ou parch sobem, a sobrevivência cai drasticamente. O "ponto ideal" de sobrevivência se concentra em núcleos pequenos (1 a 2 familiares a bordo).Separação das Classes:Região Baixa/Central (Misturadas): Nos valores 0, 1 e 2, as classes "Sobreviveu" e "Não Sobreviveu" aparecem sobrepostas no mesmo ponto, indicando que ter poucos acompanhantes não garantia a sobrevivência por si só.Região dos Extremos (Separadas): A partir de $\text{sibsp} \ge 3$ ou $\text{parch} \ge 4$, a classe "Não Sobreviveu" domina o espaço quase por completo, isolando os sobreviventes no canto inferior esquerdo do gráfico.Nota de limitação: Por serem variáveis inteiras (discretas), o overplotting no ponto $(0,0)$ esconde o fato de que a maioria absoluta dos passageiros que viajavam sozinhos não sobreviveu

#### Qualitativa x Qualitativa

#### Outliers bivariados

In [34]:
sns.jointplot(
    data=df,
    x="age",
    y="fare",
    hue="sobreviveu",
    palette={"Não Sobreviveu": COR_CONTEXTO, "Sobreviveu": COR_DESTAQUE},
    alpha=0.7,
    height=6,
)
plt.suptitle("Idade x Tarifa (fare): outliers conjuntos?", y=1.02)
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\2004852244.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Ajuste de contexto:** Adaptando a discussão para o gráfico de **`age` x `fare**`:

* **Combinações atípicas que se destacam:**
* **Idade média (~35 anos) + Tarifa máxima (`fare` > 500):** Ter cerca de 35 anos é a faixa mais comum do dataset (passa completamente despercebida na distribuição de idade). Porém, combinada à tarifa extrema de mais de $500, essa observação se isola no topo do gráfico como um outlier bivariado claro.
* **Crianças de 0 a 5 anos + Tarifa alta (`fare` > 150):** A idade em si é comum na amostra, mas a combinação com bilhetes caríssimos chama a atenção por representar famílias de altíssima renda viajando na 1ª classe.
* **Idoso extremo (~80 anos) + Tarifa baixa (`fare` < 50):** O passageiro de 80 anos já era um outlier na análise univariada de idade. O gráfico conjunto revela que ele viajava com um bilhete comum de baixo custo, sem se destacar no eixo vertical.


* **Conclusão bivariada:** A imensa maioria dos passageiros está concentrada na base do gráfico (tarifas < $100) ao longo de todas as idades. Os pontos dispersos na parte superior não são erros de digitação, mas sim o grupo restrito de passageiros das suítes de luxo.

#### Análise Multivariada

##### Matriz de Correlação

In [35]:
plt.figure(figsize=(7, 6))
sns.heatmap(df[VARS_QUANT].corr(method="spearman"), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz de Correlação (Spearman) — Variáveis Quantitativas")
plt.tight_layout()
plt.show()


C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\545335840.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Como interpretar:**


*   |r| < 0.3 ~ fraca,
*   0.3 < |r| < 0.7 ~ moderada,
*  |r| > 0.7 ~ forte.

Alguma correlação chama atenção por ser forte o suficiente pra sugerir redundância entre variáveis?

Nenhuma correlação é forte o suficiente (|r| > 0.7) para sugerir redundância ou risco de multicolinearidade.

Ausência de redundância: O maior valor na matriz é 0,42 (entre sibsp e parch), classificando-se como uma correlação moderada. Todas as quatro variáveis quantitativas preservam informação própria e independente.

Padrões de correlação moderada: As maiores associações ocorrem entre sibsp x parch (0,42), fare x sibsp (0,41) e fare x parch (0,38). Isso reflete a estrutura de viagem em grupo: passageiros com mais familiares a bordo tendem a comprar passagens juntas, inflacionando o valor acumulado da tarifa (fare).

Comportamento da idade: A variável age possui apenas correlações fracas (|r| < 0,3) com as demais, operando de forma praticamente independente do custo da passagem e do tamanho da família.

##### Correlograma

In [36]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def circulo_correlacao(x, y, **kwargs):
  # pd.Series.corr lida com NaNs com segurança
  r = pd.Series(x).corr(pd.Series(y))
  if np.isnan(r):
    r = 0.0

  ax = plt.gca()
  ax.scatter(
      [0.5],
      [0.5],
      s=abs(r) * 4000,  # Tamanho proporcional à força da correlação
      c=[r],
      cmap='coolwarm',
      vmin=-1,
      vmax=1,
      transform=ax.transAxes,
      edgecolor='white',
      linewidth=0.5,
  )
  ax.annotate(
      f'{r:.2f}',
      xy=(0.5, 0.5),
      xycoords=ax.transAxes,
      ha='center',
      va='center',
      fontsize=10,
      color='white' if abs(r) > 0.5 else 'black',
  )
  ax.set_xticks([])
  ax.set_yticks([])
  ax.grid(False)


# Construção do PairGrid
g = sns.PairGrid(df[VARS_QUANT], diag_sharey=False)
g.map_lower(sns.scatterplot, color=COR_NEUTRA, alpha=0.6, s=25)
g.map_diag(sns.histplot, color=COR_NEUTRA, kde=True)
g.map_upper(circulo_correlacao)

# Barra de cor compartilhada
sm = ScalarMappable(norm=Normalize(-1, 1), cmap='coolwarm')
sm.set_array([])
g.fig.colorbar(sm, ax=g.axes, shrink=0.6, label='Correlação (r)')

g.fig.suptitle(
    'Correlograma: Dispersão × Correlação Linear (Pearson)', y=1.02, fontsize=15
)
plt.show()

C:\Users\vagne\AppData\Local\Temp\ipykernel_25032\242142050.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Principais Conclusões**

* **Balanceamento do Alvo e Desbalanceamento de Atributos:** A variável alvo (`sobreviveu`) apresentou distribuição ideal para modelagem (entropia de 0,977 e razão de desbalanceamento de 1,44). Em contrapartida, `embarked` / `embark_town` e `deck` exibiram forte desequilíbrio e alto volume de nulos, exigindo cautela na análise bivariada.
* **Captura de Relações Não-Lineares:** A forte assimetria em `fare` fez com que a correlação de Pearson subestimasse associações. A matriz de Spearman revelou correlações moderadas reais entre `fare` x `sibsp` ($0{,}41$) e `fare` x `parch` ($0{,}38$), confirmando que o valor gasto cresce de forma monotônica com o tamanho da família.
* **Impacto do Tamanho Familiar na Sobrevivência:** Núcleos familiares pequenos ou moderados (1 a 3 acompanhantes) concentraram as maiores taxas de sobrevivência, enquanto famílias numerosas ($\text{sibsp} \ge 3$ ou $\text{parch} \ge 4$) registraram mortalidade quase total.
* **Outliers Bivariados da 1ª Classe:** Pontos atípicos extremos ($\text{fare} > 500$) concentraram-se na faixa dos ~35 anos. Esses registros não representam erros de digitação, mas sim passageiros de altíssima renda em suítes de luxo.
* **Ausência de Multicolinearidade:** Nenhuma combinação de variáveis quantitativas ultrapassou o limiar de $\vert{}r\vert{} > 0{,}7$, garantindo que `age`, `fare`, `sibsp` e `parch` preservam informações independentes para as etapas preditivas.

---

**Declaração de Uso de IA**

* **Ferramentas Utilizadas:** Gemini (Modelo de Linguagem de Larga Escala).
* **Aplicações do Modelo:** Auxílio na escrita e refatoração do código Python (Pandas, Seaborn, Scipy), cálculo automatizado de métricas de desbalanceamento/entropia de Shannon e estruturação das discussões analíticas para a documentação do projeto.


## Download dos Datasets

Vamos salvar duas versões do dataset:

- `data/raw/titanic.csv`: o dataset **original**, como veio do `seaborn` (sem tratamento).
- `data/clean/dataset.csv`: o dataset **tratado**, no estado final do `df` ao longo da EDA. É este arquivo que o `app.py` lê.

> Rode o notebook **do início ao fim** antes desta seção, para que o `df` esteja no estado final.

In [37]:
# Define as pastas de destino (funciona rodando de notebooks/ ou da raiz do projeto)
from pathlib import Path

RAIZ = Path("..") if Path("../data").exists() else Path(".")
PASTA_RAW = RAIZ / "data" / "raw"
PASTA_CLEAN = RAIZ / "data" / "clean"
PASTA_RAW.mkdir(parents=True, exist_ok=True)
PASTA_CLEAN.mkdir(parents=True, exist_ok=True)

In [38]:
# Dataset ORIGINAL (recarregado do seaborn, sem nenhuma alteração)
df_raw = sns.load_dataset("titanic")
df_raw.to_csv(PASTA_RAW / "titanic.csv", index=False)

print(f"Raw salvo em {PASTA_RAW / 'titanic.csv'} — {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas")

Raw salvo em ..\data\raw\titanic.csv — 891 linhas x 15 colunas


In [39]:
# Dataset TRATADO (estado final do `df`)
df.to_csv(PASTA_CLEAN / "dataset.csv", index=False)

print(f"Clean salvo em {PASTA_CLEAN / 'dataset.csv'} — {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"Linhas removidas no tratamento: {df_raw.shape[0] - df.shape[0]}")

Clean salvo em ..\data\clean\dataset.csv — 782 linhas x 16 colunas
Linhas removidas no tratamento: 109
